# DiffuGPT-S: General attention-head heatmap trajectories over diffusion time

Built on Raghu's `DiffuGPT_organized_Raghu.ipynb` helpers (Issues #47/#48/#49). This notebook is self-contained: it loads the model, redefines the shared helpers, and adds a **general per-head heatmap-trajectory** tool.

What it produces:
1. **L0 H0 attention heatmap trajectories** for **5 sentences**, across diffusion time 0->63. Each sentence gets a multipage PDF (one heatmap per timestep) and a compact grid PDF, in diffusion-time order, with token visibility labels `[U]/[M]` and the head's mean attention entropy in each title - so you can read *which tokens receive attention as entropy increases*.
2. A **general function** `collect_head_trajectory(text, layer, head)` + savers, so you can point it at any `(layer, head)` and get the same trajectory PDFs.

All figures are written as **PDF with Type 1 / embedded fonts** via `plt.rcParams['pdf.fonttype'] = 42` before saving. Runtime target: Colab/Jupyter with a **T4** GPU. Run cells top to bottom.

## Part 0 - Environment setup and model loading (from Raghu's notebook)

In [ ]:
!nvidia-smi

!pip install -q transformers==4.44.2 huggingface_hub

!rm -rf DiffuLLaMA && git clone --depth 1 https://github.com/HKUNLP/DiffuLLaMA.git
%cd DiffuLLaMA

import torch
from transformers import AutoConfig, AutoTokenizer
from model import DiscreteDiffusionModel, generate_samples

MODEL_NAME = "diffusionfamily/diffugpt-s"
BASE_MODEL = "gpt2"  # only the config is read from this; weights come from MODEL_NAME

config = AutoConfig.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = DiscreteDiffusionModel.from_pretrained(
    MODEL_NAME,
    model=BASE_MODEL,
    config=config,
    tokenizer=tokenizer,
    device="cuda",
).to("cuda")
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_NAME}: {n_params/1e6:.1f}M params, hidden={config.hidden_size}, layers={config.num_hidden_layers}")


## Shared helpers (from Raghu's notebook)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["pdf.fonttype"] = 42  # Embed fonts in PDF outputs for reproducible/publication-friendly figures.
plt.rcParams["ps.fonttype"] = 42
import pandas as pd
from model import get_anneal_attn_mask

@torch.no_grad()
def forward_with_attentions(model, input_ids, attention_mask):
    """
    Runs one DiffuGPT denoising forward pass and returns:
    - logits
    - attentions from every layer

    attentions[layer] shape:
        [batch, num_heads, seq_len, seq_len]
    """
    x_embed = model.get_embeds(input_ids)

    outputs = model.denoise_model(
        inputs_embeds=x_embed,
        attention_mask=attention_mask,
        output_attentions=True,
        output_hidden_states=False,
        return_dict=True,
        use_cache=False,
    )

    logits = model.get_logits(outputs.last_hidden_state)
    attentions = outputs.attentions

    return logits, attentions

text_sequence = "Today is a wonderful day,"

def tokenize_for_experiment(tokenizer, text_sequence, add_bos=True):
    token_ids = tokenizer.encode(text_sequence)

    if add_bos:
        token_ids = [tokenizer.bos_token_id] + token_ids

    input_ids = torch.tensor([token_ids], device=model.device)

    readable_tokens = []
    for tok_id in token_ids:
        readable_tokens.append(tokenizer.decode([tok_id]))

    return input_ids, readable_tokens

true_input_ids, readable_tokens = tokenize_for_experiment(tokenizer, text_sequence)

print("seq_len:", true_input_ids.shape[1])
for i, tok in enumerate(readable_tokens):
    print(i, repr(tok))



In [ ]:
@torch.no_grad()
def get_xt_at_diffusion_time(
    model,
    tokenizer,
    text_sequence,
    diffusion_time,
    diffusion_steps=64,
    seed=42,
    include_bos=True,
    force_final_fully_unmasked=True,
):
    """
    Returns the token state xt at a chosen diffusion time.

    diffusion_time:
        0  = beginning, mostly masked
        63 = final step, fully unmasked if force_final_fully_unmasked=True

    Returns:
        xt: token IDs at this diffusion time
        readable_tokens: token labels
        is_visible: boolean list, True if token is currently unmasked/visible
        unmask_step: step when each token became visible
    """

    assert 0 <= diffusion_time < diffusion_steps

    torch.manual_seed(seed)
    np.random.seed(seed)

    true_input_ids, readable_tokens = tokenize_for_experiment(
        tokenizer,
        text_sequence,
        add_bos=include_bos,
    )

    true_input_ids = true_input_ids.to(model.device)
    batch_size, seq_len = true_input_ids.shape

    if tokenizer.mask_token_id is None:
        raise ValueError("tokenizer.mask_token_id is None.")

    # Start from the true sentence
    xt = true_input_ids.clone()

    # Mask every token except BOS
    maskable_mask = torch.ones_like(xt, dtype=torch.bool)

    if include_bos:
        maskable_mask[:, 0] = False

    xt = xt.masked_fill(maskable_mask, tokenizer.mask_token_id)

    remaining_mask = maskable_mask.clone()

    unmask_step = [-1] * seq_len

    if include_bos:
        unmask_step[0] = 0

    # Reproduce teacher-forced random unmasking schedule
    for progress_step in range(diffusion_time):
        diffusion_t = diffusion_steps - progress_step
        p_to_unmask = 1.0 / diffusion_t

        reveal_now = remaining_mask & (
            torch.rand_like(
                remaining_mask,
                dtype=torch.float,
                device=model.device,
            ) < p_to_unmask
        )

        xt = xt.clone()
        xt[reveal_now] = true_input_ids[reveal_now]

        reveal_positions = reveal_now[0].nonzero(as_tuple=True)[0].tolist()

        for pos in reveal_positions:
            if unmask_step[pos] == -1:
                unmask_step[pos] = progress_step + 1

        remaining_mask = remaining_mask & (~reveal_now)

    # For final heatmaps, force the sentence to be fully unmasked
    if force_final_fully_unmasked and diffusion_time == diffusion_steps - 1:
        xt = true_input_ids.clone()
        remaining_mask = torch.zeros_like(remaining_mask, dtype=torch.bool)

        for pos in range(seq_len):
            if unmask_step[pos] == -1:
                unmask_step[pos] = diffusion_time

    is_visible = (~remaining_mask[0]).detach().cpu().tolist()

    return xt, readable_tokens, is_visible, unmask_step



In [ ]:
@torch.no_grad()
def get_all_attention_matrices_at_time(
    model,
    tokenizer,
    text_sequence,
    diffusion_time,
    diffusion_steps=64,
    seed=42,
    include_bos=True,

):
    """
    Gets all attention matrices at a specific diffusion time.

    Returns:
        attentions:
            tuple of length num_layers
            attentions[layer].shape = [batch, num_heads, seq_len, seq_len]

        readable_tokens:
            token labels

        is_visible:
            which tokens are visible/unmasked at this diffusion time

        unmask_step:
            when each token became visible
    """

    xt, readable_tokens, is_visible, unmask_step = get_xt_at_diffusion_time(
        model=model,
        tokenizer=tokenizer,
        text_sequence=text_sequence,
        diffusion_time=diffusion_time,
        diffusion_steps=diffusion_steps,
        seed=seed,
        include_bos=include_bos,
        force_final_fully_unmasked=True,
    )

    batch_size, seq_len = xt.shape
    x_embed = model.get_embeds(xt)

    attention_mask = get_anneal_attn_mask(
        seq_len=seq_len,
        bsz=batch_size,
        dtype=x_embed.dtype,
        device=xt.device,
        attn_mask_ratio=1.0,
    )

    logits, attentions = forward_with_attentions(
        model=model,
        input_ids=xt,
        attention_mask=attention_mask,
    )

    return attentions, readable_tokens, is_visible, unmask_step



In [ ]:
def plot_single_attention_heatmap(
    attn_matrix,
    readable_tokens,
    layer_idx,
    head_idx,
    diffusion_time,
    is_visible=None,
    drop_bos=False,
    renormalize_rows=False,
    title=None,
):
    """
    Plots token-by-token attention heatmap.

    Rows = target tokens doing the attending.
    Columns = source tokens being attended to.

    drop_bos:
        If True, removes token 0, usually <|endoftext|>/BOS,
        from both rows and columns.

    renormalize_rows:
        If True, rescales each row after removing BOS so the remaining
        word-to-word attention sums to 1.
    """

    attn_np = attn_matrix.detach().float().cpu().numpy()

    tokens = list(readable_tokens)

    if is_visible is not None:
        visible = list(is_visible)
    else:
        visible = None

    # Optional: remove BOS row and BOS column
    if drop_bos:
        attn_np = attn_np[1:, 1:]
        tokens = tokens[1:]

        if visible is not None:
            visible = visible[1:]

    # Optional: renormalize rows after dropping BOS
    if renormalize_rows:
        row_sums = attn_np.sum(axis=-1, keepdims=True)
        attn_np = attn_np / (row_sums + 1e-12)

    # Build labels
    token_labels = []

    for i, tok in enumerate(tokens):
        if visible is None:
            token_labels.append(tok)
        else:
            status = "U" if visible[i] else "M"
            token_labels.append(f"{tok} [{status}]")

    plt.figure(figsize=(9, 7))
    plt.imshow(attn_np, aspect="auto", interpolation="nearest")

    plt.xticks(
        ticks=np.arange(len(token_labels)),
        labels=token_labels,
        rotation=90,
    )

    plt.yticks(
        ticks=np.arange(len(token_labels)),
        labels=token_labels,
    )

    plt.xlabel("Source token attended to")
    plt.ylabel("Target token attending")
    plt.colorbar(label="Attention weight")

    if title is None:
        bos_text = "No BOS" if drop_bos else "With BOS"
        title = f"{bos_text} attention heatmap | Layer {layer_idx}, Head {head_idx}, Time {diffusion_time}"

    plt.title(title)
    plt.tight_layout()
    plt.show()



In [ ]:
@torch.no_grad()
def collect_attention_entropy_over_time(
    model,
    tokenizer,
    text_sequence,
    layer_idx=0,
    head_idx=0,
    diffusion_steps=64,
    seed=42,
    include_bos=True,
    normalize_entropy=False,
):
    """
    Measures attention entropy over diffusion time.

    Args:
        text_sequence:
            The sentence you want to analyze.

        layer_idx:
            Which transformer layer to inspect.
            DiffuGPT-S has 12 layers, so use 0 through 11.

        head_idx:
            Which attention head to inspect.

        diffusion_steps:
            Usually 64, matching your notebook.

        seed:
            Controls the random unmasking schedule.

        normalize_entropy:
            If False, entropy is in natural-log units.
            If True, entropy is divided by log(seq_len), so values are 0 to 1.

    Returns:
        entropy_matrix:
            shape [diffusion_steps, seq_len]

        readable_tokens:
            list of token strings

        unmask_step:
            list where unmask_step[i] is the diffusion progress step when token i became visible.
            BOS is visible from the start.
    """

    torch.manual_seed(seed)
    np.random.seed(seed)

    # -----------------------------
    # Tokenize
    # -----------------------------
    true_input_ids, readable_tokens = tokenize_for_experiment(
        tokenizer,
        text_sequence,
        add_bos=include_bos,
    )

    true_input_ids = true_input_ids.to(model.device)
    batch_size, seq_len = true_input_ids.shape

    # -----------------------------
    # Start with all tokens masked except BOS
    # -----------------------------
    xt = true_input_ids.clone()

    if tokenizer.mask_token_id is None:
        raise ValueError("tokenizer.mask_token_id is None. DiffuGPT should have a mask token.")

    maskable_mask = torch.ones_like(xt, dtype=torch.bool)

    if include_bos:
        maskable_mask[:, 0] = False

    xt = xt.masked_fill(maskable_mask, tokenizer.mask_token_id)

    # Track when each token becomes unmasked
    unmask_step = [-1] * seq_len
    if include_bos:
        unmask_step[0] = 0

    # -----------------------------
    # Build DiffuGPT 4D attention mask
    # -----------------------------
    x_embed = model.get_embeds(xt)

    attention_mask = get_anneal_attn_mask(
        seq_len=seq_len,
        bsz=batch_size,
        dtype=x_embed.dtype,
        device=xt.device,
        attn_mask_ratio=1.0,
    )

    # -----------------------------
    # Storage
    # -----------------------------
    all_attention_mats = []
    all_xt_states = []

    # progress_step goes 0 -> diffusion_steps - 1
    # progress_step = 0 means most masked
    # progress_step = 63 means late denoising
    remaining_mask = maskable_mask.clone()

    for progress_step in range(diffusion_steps):
        diffusion_t = diffusion_steps - progress_step

        logits, attentions = forward_with_attentions(
            model=model,
            input_ids=xt,
            attention_mask=attention_mask,
        )

        if attentions is None:
            raise RuntimeError(
                "No attentions were returned. Make sure output_attentions=True is being passed."
            )

        # attentions[layer_idx]: [batch, heads, seq, seq]
        attn = attentions[layer_idx][0, head_idx].detach().float().cpu()

        # Store [seq_len, seq_len]
        all_attention_mats.append(attn)
        all_xt_states.append(xt.detach().cpu().clone())

        # Stop after final measurement
        if progress_step == diffusion_steps - 1:
            break

        # -----------------------------
        # Reveal some still-masked tokens.
        # This mirrors the DiffuGPT random denoising schedule.
        # -----------------------------
        p_to_unmask = 1.0 / diffusion_t

        reveal_now = remaining_mask & (
            torch.rand_like(remaining_mask, dtype=torch.float, device=model.device) < p_to_unmask
        )

        xt = xt.clone()
        xt[reveal_now] = true_input_ids[reveal_now]

        # Record the progress step when each token became visible
        reveal_positions = reveal_now[0].nonzero(as_tuple=True)[0].tolist()
        for pos in reveal_positions:
            if unmask_step[pos] == -1:
                unmask_step[pos] = progress_step + 1

        remaining_mask = remaining_mask & (~reveal_now)

    # -----------------------------
    # Convert attention to entropy
    # -----------------------------
    attention_over_time = torch.stack(all_attention_mats, dim=0)
    # shape: [diffusion_steps, seq_len, seq_len]

    probs = attention_over_time.clamp_min(1e-12)
    entropy_matrix = -(probs * probs.log()).sum(dim=-1)
    # shape: [diffusion_steps, seq_len]

    if normalize_entropy:
        entropy_matrix = entropy_matrix / np.log(seq_len)

    entropy_matrix = entropy_matrix.numpy()

    return entropy_matrix, readable_tokens, unmask_step, attention_over_time.numpy(), all_xt_states


## General per-head heatmap-trajectory tool

`collect_head_trajectory` runs one denoising trajectory for a chosen `(layer, head)` and returns the attention `[steps, S, S]` plus per-step token visibility. The savers render heatmaps for any subset of diffusion timesteps as Type-1-font PDFs.

Rows = target token attending; columns = source token attended to. Each title reports the head's mean attention entropy at that timestep (BOS row excluded), so increasing entropy = attention spreading across more source tokens.

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

plt.rcParams["pdf.fonttype"] = 42   # Type 1 / embedded fonts in PDF outputs
plt.rcParams["ps.fonttype"] = 42

def _subset_timesteps(diffusion_steps, n):
    if n >= diffusion_steps:
        return list(range(diffusion_steps))
    idx = np.linspace(0, diffusion_steps - 1, n).round().astype(int)
    return sorted(set(int(i) for i in idx))

def _labels_with_status(tokens, is_visible):
    return [f"{tok} [{'U' if vis else 'M'}]" for tok, vis in zip(tokens, is_visible)]

def _row_entropy_mean(attn_np, skip_bos=True):
    p = np.clip(np.asarray(attn_np), 1e-12, None)
    ent = -(p * np.log(p)).sum(axis=-1)
    if skip_bos and len(ent) > 1:
        ent = ent[1:]
    return float(ent.mean())

def _prep(attn_np, tokens, is_visible, drop_bos, renorm):
    attn_np = np.asarray(attn_np)
    tokens = list(tokens); vis = list(is_visible)
    if drop_bos:
        attn_np = attn_np[1:, 1:]; tokens = tokens[1:]; vis = vis[1:]
    if renorm:
        rs = attn_np.sum(axis=-1, keepdims=True); attn_np = attn_np / (rs + 1e-12)
    return attn_np, _labels_with_status(tokens, vis)

def _draw_heatmap(ax, attn_np, token_labels, fontsize=6):
    im = ax.imshow(attn_np, aspect="auto", interpolation="nearest")
    ax.set_xticks(np.arange(len(token_labels)))
    ax.set_xticklabels(token_labels, rotation=90, fontsize=fontsize)
    ax.set_yticks(np.arange(len(token_labels)))
    ax.set_yticklabels(token_labels, fontsize=fontsize)
    return im

def collect_head_trajectory(text, layer_idx, head_idx, seed=42, diffusion_steps=64):
    """One denoising run for (layer, head): tokens, attn[steps,S,S], visibility[steps][S], unmask_step."""
    _, toks, unmask_step, attn_over_time, xt_states = collect_attention_entropy_over_time(
        model=model, tokenizer=tokenizer, text_sequence=text,
        layer_idx=layer_idx, head_idx=head_idx,
        diffusion_steps=diffusion_steps, seed=seed, include_bos=True,
    )
    mask_id = int(tokenizer.mask_token_id)
    vis_over_time = [[int(t) != mask_id for t in xt[0].tolist()] for xt in xt_states]
    return toks, attn_over_time, vis_over_time, unmask_step

def save_head_trajectory_pages(text, layer_idx, head_idx, timesteps, out_path,
                               toks, attn_over_time, vis_over_time,
                               diffusion_steps=64, drop_bos=False, renorm=False):
    with PdfPages(out_path) as pdf:
        for t in timesteps:
            a, labels = _prep(attn_over_time[t], toks, vis_over_time[t], drop_bos, renorm)
            Hbar = _row_entropy_mean(attn_over_time[t])
            fig, ax = plt.subplots(figsize=(9, 7))
            im = _draw_heatmap(ax, a, labels)
            ax.set_xlabel("Source token attended to"); ax.set_ylabel("Target token attending")
            fig.colorbar(im, ax=ax, label="Attention weight")
            ax.set_title(f"L{layer_idx} H{head_idx} | t={t}/{diffusion_steps - 1} | mean entropy={Hbar:.2f}\n{text!r}", fontsize=9)
            fig.tight_layout()
            pdf.savefig(fig); plt.close(fig)
    return out_path

def save_head_trajectory_grid(text, layer_idx, head_idx, timesteps, out_path,
                              toks, attn_over_time, vis_over_time,
                              ncols=5, diffusion_steps=64, drop_bos=False, renorm=False):
    n = len(timesteps); nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.3 * ncols, 3.1 * nrows))
    axes = np.array(axes).reshape(-1); im = None
    for k, t in enumerate(timesteps):
        a, labels = _prep(attn_over_time[t], toks, vis_over_time[t], drop_bos, renorm)
        im = _draw_heatmap(axes[k], a, labels, fontsize=5)
        axes[k].set_title(f"t={t} | H={_row_entropy_mean(attn_over_time[t]):.2f}", fontsize=7)
    for k in range(n, len(axes)):
        axes[k].axis("off")
    bos = "no-BOS, renorm" if (drop_bos and renorm) else ("no-BOS" if drop_bos else "with BOS")
    fig.suptitle(f"L{layer_idx} H{head_idx} attention over diffusion time ({bos}) | {text!r}", fontsize=10)
    if im is not None:
        fig.colorbar(im, ax=axes.tolist(), shrink=0.6, label="Attention weight")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(out_path, format="pdf"); plt.close(fig)
    return out_path

print("trajectory tools ready")

## Run: L0 H0 heatmap trajectories for 5 sentences

Set `FULL_64 = True` to emit all 64 timesteps per page-PDF; otherwise a 10-timestep subset (evenly spaced 0->63) is used. Change `TRAJ_LAYER` / `TRAJ_HEAD` to target any other head with the same code.

In [ ]:
TRAJ_LAYER = 0
TRAJ_HEAD = 0
TRAJ_SEED = 42
DIFFUSION_STEPS = 64
N_SUBSET = 10
FULL_64 = False
OUT_DIR = "attention_trajectories"
os.makedirs(OUT_DIR, exist_ok=True)

TRAJ_SENTENCES = [
    "Today is a wonderful day,",
    "The student solved the difficult problem correctly.",
    "The doctor gave the patient a careful diagnosis.",
    "The cat sat quietly on the warm windowsill.",
    "She carefully painted the old wooden fence yesterday.",
]

timesteps = _subset_timesteps(DIFFUSION_STEPS, DIFFUSION_STEPS if FULL_64 else N_SUBSET)
print("Diffusion timesteps shown:", timesteps)

saved = []
for si, text in enumerate(TRAJ_SENTENCES):
    toks, attn_over_time, vis_over_time, unmask_step = collect_head_trajectory(
        text, TRAJ_LAYER, TRAJ_HEAD, seed=TRAJ_SEED, diffusion_steps=DIFFUSION_STEPS)
    base = f"{OUT_DIR}/L{TRAJ_LAYER}_H{TRAJ_HEAD}_sent{si}"
    p_pages = save_head_trajectory_pages(text, TRAJ_LAYER, TRAJ_HEAD, timesteps,
        f"{base}_pages.pdf", toks, attn_over_time, vis_over_time,
        diffusion_steps=DIFFUSION_STEPS, drop_bos=False, renorm=False)
    p_grid = save_head_trajectory_grid(text, TRAJ_LAYER, TRAJ_HEAD, timesteps,
        f"{base}_grid_withbos.pdf", toks, attn_over_time, vis_over_time,
        diffusion_steps=DIFFUSION_STEPS, drop_bos=False, renorm=False)
    p_grid_nb = save_head_trajectory_grid(text, TRAJ_LAYER, TRAJ_HEAD, timesteps,
        f"{base}_grid_nobos.pdf", toks, attn_over_time, vis_over_time,
        diffusion_steps=DIFFUSION_STEPS, drop_bos=True, renorm=True)
    saved += [p_pages, p_grid, p_grid_nb]
    print(f"[{si}] saved 3 PDFs for: {text!r}")

print("\nAll files:")
print("\n".join(saved))

### Quick inline preview (sentence 0, with-BOS grid)

In [ ]:
_text = TRAJ_SENTENCES[0]
_toks, _attn, _vis, _ = collect_head_trajectory(_text, TRAJ_LAYER, TRAJ_HEAD, seed=TRAJ_SEED, diffusion_steps=DIFFUSION_STEPS)
_n = len(timesteps); _ncols = 5; _nrows = int(np.ceil(_n / _ncols))
fig, axes = plt.subplots(_nrows, _ncols, figsize=(3.3 * _ncols, 3.1 * _nrows))
axes = np.array(axes).reshape(-1); im = None
for k, t in enumerate(timesteps):
    a, labels = _prep(_attn[t], _toks, _vis[t], drop_bos=False, renorm=False)
    im = _draw_heatmap(axes[k], a, labels, fontsize=5)
    axes[k].set_title(f"t={t} | H={_row_entropy_mean(_attn[t]):.2f}", fontsize=7)
for k in range(_n, len(axes)):
    axes[k].axis("off")
fig.suptitle(f"L{TRAJ_LAYER} H{TRAJ_HEAD} over diffusion time | {_text!r}", fontsize=10)
fig.colorbar(im, ax=axes.tolist(), shrink=0.6, label="Attention weight")
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## Download all PDFs (Colab)

In [ ]:
import shutil
shutil.make_archive("attention_trajectories", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("attention_trajectories.zip")
except Exception as exc:
    print("Download helper skipped (local Jupyter? use the file browser):", exc)